# Test de personalidad
## 04 · Modelado

Proyecto de Sistemas Inteligentes · UNAM · Matemáticas Aplicadas y Computación

## Objetivo

Probar **varios modelos** que intenten predecir si una persona es **Introvertida** o **No introvertida** a partir de las 9 variables del cuaderno 03, y **compararlos con métricas adecuadas**.

Como además queríamos explorar sin etiqueta, al final probamos un modelo **no supervisado** (agrupamiento) para ver si los datos forman grupos parecidos a los perfiles.

## Datos

Usamos `Datos_Modelado.xlsx` (lo genera el cuaderno 03). Quedan **21 personas** con perfil: 14 Introvertidas y 7 No introvertidas.

## Cuidados que tomamos (con tan pocos datos)

- **Validación "dejar uno fuera" (leave-one-out):** el modelo se entrena con todas las personas menos una y predice a esa persona; se repite 21 veces. Así cada persona se evalúa sin que el modelo la haya visto.
- **Los faltantes se rellenan dentro de cada entrenamiento**, solo con los datos de entrenamiento (mediana o valor más común), para que no se "filtre" información de la persona evaluada.
- **No ajustamos los modelos probando muchas configuraciones.** Con 21 personas eso solo lograría que se acomoden al azar. Usamos configuraciones sencillas y fijas.
- **Comparamos contra un modelo "base"** que siempre responde la categoría más común (Introvertido). Si un modelo no supera a la base, no aprendió nada útil.
- **Prueba de azar (permutaciones):** barajamos las etiquetas muchas veces para ver con qué frecuencia el azar logra un resultado igual o mejor.

## Cómo leer las métricas

| Métrica | Qué significa |
|---|---|
| Aciertos | % de personas bien clasificadas. Engañosa si hay desbalance |
| Aciertos balanceados | Promedio de aciertos en cada perfil. **0.50 = azar** |
| Sensibilidad | De las personas realmente *No introvertidas*, cuántas detecta |
| Precisión | De las que el modelo llama *No introvertidas*, cuántas lo son |
| F1 | Combina precisión y sensibilidad |
| AUC | 0.50 = azar, 1.00 = perfecto |

La clase que queremos detectar (la "positiva") es **No introvertido**, porque es la minoritaria.

### Preparar el entorno
*Qué hace:* importa las herramientas de modelado y revisa si estamos en Google Colab.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, silhouette_score,
)
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from google.colab import files  # solo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

import sklearn
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| Colab:", EN_COLAB)

### Cargar los datos
*Qué hace:* busca `Datos_Modelado.xlsx` (si no está, te pide subirlo) y lo lee.

In [ ]:
ARCHIVO = "Datos_Modelado.xlsx"

if not os.path.exists(ARCHIVO):
    if EN_COLAB:
        print("Elige el archivo Datos_Modelado.xlsx en la ventana que se abre...")
        files.upload()
    if not os.path.exists(ARCHIVO):
        raise FileNotFoundError(
            f"No se encontró '{ARCHIVO}'. Súbelo al panel de archivos de Colab con ese nombre exacto."
        )

df = pd.read_excel(ARCHIVO, sheet_name="DatosModelado")
print(f"Personas: {df.shape[0]} | Columnas: {df.shape[1]}")

### Preparar los datos para el modelo
*Qué hace:* deja solo a las personas con perfil, separa las variables para predecir (`X`) de lo que queremos predecir (`y`, donde **1 = No introvertido**) y calcula qué porcentaje acertaría un modelo que siempre dijera "Introvertido".

Si el equipo decide **excluir a alguien** de los datos (por ejemplo, por una respuesta dudosa), se escribe su código en `EXCLUIR_PERSONAS`, por ejemplo `["U022"]`, y se vuelve a ejecutar todo el cuaderno.

In [ ]:
EXCLUIR_PERSONAS = []  # ejemplo: ["U022"]

VARIABLES = [
    "genero_femenino", "mus_tiempo_horas", "tiene_pareja",
    "rol_lider", "reaccion_incomodidad", "problema_con_otros",
    "act_conversar", "usa_spotify", "usa_youtube",
]
CONTINUAS = ["mus_tiempo_horas"]
BINARIAS = [v for v in VARIABLES if v not in CONTINUAS]

datos = df[df["perfil"].isin(["Introvertido", "No introvertido"]) & ~df["id_persona"].isin(EXCLUIR_PERSONAS)]
datos = datos.reset_index(drop=True)

X = datos[VARIABLES]
y = (datos["perfil"] == "No introvertido").astype(int)

print("Personas en el modelado:", len(datos))
print(datos["perfil"].value_counts().to_string())
base = max(y.mean(), 1 - y.mean())
print(f"\nUn modelo que siempre dijera la categoría más común acertaría {base:.0%}")

### Definir los modelos
*Qué hace:* arma cada modelo como una "tubería" (pipeline): primero rellena los faltantes y escala, y luego aplica el modelo. Probamos cuatro modelos distintos más el modelo base.

- **Base:** siempre responde la categoría más común.
- **Regresión logística:** suma el peso de cada variable (es el más fácil de explicar).
- **Árbol de decisión (profundidad 2):** hace preguntas sencillas de sí/no.
- **Bosque aleatorio:** muchos árboles pequeños que votan.
- **k vecinos (k = 3):** compara a cada persona con las 3 más parecidas.

In [ ]:
def preprocesador(escalar):
    continuas = [("imputar", SimpleImputer(strategy="median"))]
    if escalar:
        continuas.append(("escalar", StandardScaler()))
    return ColumnTransformer([
        ("continuas", Pipeline(continuas), CONTINUAS),
        ("binarias", SimpleImputer(strategy="most_frequent"), BINARIAS),
    ])


def tuberia(modelo, escalar):
    return Pipeline([("prep", preprocesador(escalar)), ("modelo", modelo)])


MODELOS = {
    "Base (siempre la categoría común)": tuberia(DummyClassifier(strategy="most_frequent"), False),
    "Regresión logística": tuberia(LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000), True),
    "Árbol de decisión (prof. 2)": tuberia(DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=0), False),
    "Bosque aleatorio": tuberia(RandomForestClassifier(n_estimators=50, max_depth=3, class_weight="balanced", random_state=0), False),
    "k vecinos (k = 3)": tuberia(KNeighborsClassifier(n_neighbors=3), True),
}
print("Modelos definidos:", list(MODELOS))

## Comparar los modelos
*Qué hace:* evalúa cada modelo con la validación "dejar uno fuera" y calcula las métricas. La tabla queda ordenada por **aciertos balanceados**.

In [ ]:
def evaluar(modelo, X, y):
    proba = cross_val_predict(modelo, X, y, cv=LeaveOneOut(), method="predict_proba")[:, 1]
    pred = (proba >= 0.5).astype(int)
    return pred, proba


filas, predicciones = [], {}
for nombre, modelo in MODELOS.items():
    pred, proba = evaluar(modelo, X, y)
    predicciones[nombre] = pred
    filas.append({
        "modelo": nombre,
        "aciertos": accuracy_score(y, pred),
        "aciertos_balanceados": balanced_accuracy_score(y, pred),
        "sensibilidad": recall_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "auc": roc_auc_score(y, proba),
    })

resultados = pd.DataFrame(filas).set_index("modelo").sort_values("aciertos_balanceados", ascending=False)
display(resultados.round(2))

### Gráfica de comparación
*Qué hace:* dibuja los aciertos balanceados de cada modelo. La línea punteada en **0.50** es el azar: un modelo que no la supera no sirve.

In [ ]:
orden = resultados["aciertos_balanceados"].sort_values()
plt.figure(figsize=(8, 4))
plt.barh(orden.index, orden.values)
plt.axvline(0.5, linestyle="--", color="gray")
plt.xlim(0, 1)
plt.xlabel("Aciertos balanceados (0.50 = azar)")
plt.title("Comparación de modelos (dejar uno fuera)")
plt.tight_layout()
plt.show()

### Matriz de confusión del mejor modelo
*Qué hace:* toma el mejor modelo (sin contar el modelo base) y muestra cuántas personas acertó y en qué se equivocó. Filas = lo que la persona realmente es; columnas = lo que el modelo dijo.

In [ ]:
candidatos = resultados.drop(index="Base (siempre la categoría común)")
mejor = candidatos.index[0]
print("Mejor modelo por aciertos balanceados:", mejor)

matriz = confusion_matrix(y, predicciones[mejor])
etiquetas = ["Introvertido", "No introvertido"]
fig, eje = plt.subplots(figsize=(4.5, 4))
eje.imshow(matriz, cmap="Blues")
eje.set_xticks([0, 1])
eje.set_yticks([0, 1])
eje.set_xticklabels(etiquetas)
eje.set_yticklabels(etiquetas)
eje.set_xlabel("Lo que dijo el modelo")
eje.set_ylabel("Lo que realmente es")
for i in range(2):
    for j in range(2):
        eje.text(j, i, int(matriz[i, j]), ha="center", va="center", fontsize=14)
plt.title(mejor, fontsize=10)
plt.tight_layout()
plt.show()

## ¿El resultado puede ser casualidad?
*Qué hace:* barajamos las etiquetas (quién es introvertido y quién no) 100 veces, y cada vez repetimos todo el proceso con el mejor modelo. El **valor p** es la proporción de veces que el azar logró igualar o superar el resultado real. Un valor p alto (por ejemplo, mayor a 0.10) significa que **no podemos descartar que el resultado sea casualidad**.

Ojo: escogimos el mejor de cuatro modelos, así que este valor p es algo optimista.

In [ ]:
def valor_p_permutacion(modelo, X, y, repeticiones=100, semilla=0):
    generador = np.random.default_rng(semilla)
    pred, _ = evaluar(modelo, X, y)
    real = balanced_accuracy_score(y, pred)
    iguales_o_mejores = 0
    for _ in range(repeticiones):
        y_barajada = generador.permutation(y.to_numpy())
        pred_b, _ = evaluar(modelo, X, y_barajada)
        if balanced_accuracy_score(y_barajada, pred_b) >= real:
            iguales_o_mejores += 1
    return real, (iguales_o_mejores + 1) / (repeticiones + 1)


real, p = valor_p_permutacion(MODELOS[mejor], X, y)
print(f"{mejor}: aciertos balanceados reales = {real:.2f}  |  valor p por azar = {p:.2f}")

## ¿Qué variables pesan más?
*Qué hace:* entrena la **regresión logística** con todas las personas y muestra el peso de cada variable (positivo = empuja hacia *No introvertido*; negativo = hacia *Introvertido*). Sirve para **entender**, no para demostrar nada: con 21 personas los pesos cambian mucho de una muestra a otra.

In [ ]:
logistica = MODELOS["Regresión logística"].fit(X, y)
pesos = pd.Series(logistica.named_steps["modelo"].coef_[0], index=CONTINUAS + BINARIAS).sort_values()

plt.figure(figsize=(7, 4))
plt.barh(pesos.index, pesos.values)
plt.axvline(0, color="gray")
plt.xlabel("Peso en la regresión logística")
plt.title("Variables que empujan hacia cada perfil")
plt.tight_layout()
plt.show()
display(pesos.round(2).to_frame("peso"))

## Modelo no supervisado: agrupamiento (K-means)
*Qué hace:* sin usar el perfil, pide al modelo que forme **2 grupos** de personas parecidas según las 9 variables. Luego comparamos esos grupos con los perfiles. El **índice de Rand ajustado** vale 0 si los grupos no tienen relación con los perfiles y 1 si coinciden perfectamente.

In [ ]:
Xk = preprocesador(True).fit_transform(X)
agrupador = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xk)

cruce = pd.crosstab(pd.Series(agrupador.labels_, name="grupo"), datos["perfil"])
display(cruce)
print("Índice de Rand ajustado:", round(adjusted_rand_score(y, agrupador.labels_), 2))
print("Silueta (qué tan separados están los grupos, de -1 a 1):", round(silhouette_score(Xk, agrupador.labels_), 2))

### Guardar los resultados
*Qué hace:* guarda en `Resultados_Modelos.xlsx` la comparación de modelos, la matriz de confusión, los pesos de la regresión logística y un resumen (prueba de azar y agrupamiento). La base de datos y el tablero usan este archivo.

In [ ]:
resumen = pd.DataFrame({
    "concepto": [
        "personas_modelado", "introvertidos", "no_introvertidos", "mejor_modelo", "aciertos_balanceados_mejor",
        "valor_p_azar", "repeticiones_prueba_azar", "rand_ajustado_kmeans", "silueta_kmeans",
    ],
    "valor": [
        len(datos), int((y == 0).sum()), int((y == 1).sum()), mejor, round(float(real), 4),
        round(float(p), 4), 100, round(float(adjusted_rand_score(y, agrupador.labels_)), 4),
        round(float(silhouette_score(Xk, agrupador.labels_)), 4),
    ],
})
resumen["valor"] = resumen["valor"].astype(str)

SALIDA = "Resultados_Modelos.xlsx"
with pd.ExcelWriter(SALIDA) as escritor:
    resultados.reset_index().to_excel(escritor, sheet_name="Modelos", index=False)
    pd.DataFrame(matriz, index=etiquetas, columns=etiquetas).rename_axis("real").reset_index().to_excel(
        escritor, sheet_name="MatrizConfusion", index=False)
    pesos.rename_axis("variable").reset_index(name="peso").to_excel(escritor, sheet_name="PesosLogistica", index=False)
    resumen.to_excel(escritor, sheet_name="Resumen", index=False)

print("Guardado:", SALIDA)
if EN_COLAB:
    files.download(SALIDA)

## Conclusiones

**Resultados (21 personas, validación "dejar uno fuera")**

| Modelo | Aciertos | Aciertos balanceados | F1 | AUC |
|---|---|---|---|---|
| Árbol de decisión (prof. 2) | 0.67 | 0.68 | 0.59 | 0.54 |
| Bosque aleatorio | 0.67 | 0.61 | 0.46 | 0.68 |
| Base (siempre "Introvertido") | 0.67 | 0.50 | 0.00 | 0.50 |
| k vecinos (k = 3) | 0.48 | 0.43 | 0.27 | 0.45 |
| Regresión logística | 0.29 | 0.21 | 0.00 | 0.15 |

- **Ningún modelo supera de forma convincente al modelo base ni al azar.** El mejor (árbol de decisión) llega a 0.68 de aciertos balanceados, pero la prueba de permutaciones da un **valor p de 0.23**: el azar logra un resultado igual o mejor casi una de cada cuatro veces.
- La regresión logística quedó **por debajo del azar**. Con 21 personas y 9 variables esto es normal: el modelo se acomoda al ruido de los datos de entrenamiento y no generaliza.
- El agrupamiento (K-means) tampoco encontró los perfiles: el índice de Rand ajustado es **-0.06** (sin relación) y la silueta es baja (0.20).
- Los pesos de la regresión logística apuntan en una dirección razonable (ser líder y disfrutar platicar empujan hacia *No introvertido*), pero son **inestables** y no se pueden tomar como evidencia.

**Qué significa**

Con estas 9 variables y 21 personas **no hay evidencia de que se pueda predecir el perfil**. Es un resultado válido, y es mejor reportarlo con honestidad que forzar un modelo que parezca bueno.

**Por qué creemos que pasó**

- **Muy pocas personas** y un perfil minoritario de solo 7.
- **El perfil se definió con solo dos respuestas**, y las demás variables miden cosas distintas.
- **Las variables salen de respuestas de texto libre convertidas con reglas sencillas**, que pierden matices.
- El grupo es **muy homogéneo** (misma carrera y semestre) y las respuestas son autodescripciones.

**Qué recomendaríamos para un proyecto real**

1. Reunir **muchas más personas** (cientos, no decenas).
2. Usar un **test de personalidad validado** para definir el perfil, en lugar de dos preguntas.
3. Diseñar preguntas **cerradas** (escalas de 1 a 5) que no necesiten convertir texto.
4. Volver a probar con **validación cruzada y prueba de azar**, como hicimos aquí.